# Deliverable 2: Risk Analysis, Simulation and Investment Strategy

Capital Analyst — Evidence 2
Camila Licona Toledo — A00838736

This notebook continues from Deliverable 1. We keep the same fund of 10 stocks
(5 Mexican and 5 from the US) and the same client, George Torres, 41 years old,
with a 10+ year horizon, moderate risk tolerance and capital appreciation as his
main objective.

Here we do six things:

1. Justify the investment universe
2. Calculate the risk metrics: volatility, Beta, Sharpe and VaR
3. Measure the goodness of fit of the model with R2, AIC and BIC
4. Simulate the portfolio and backtest a strategy
5. Define the strategy and write the script that generates the orders
6. Build the final visualizations

## 0. Libraries

In [ ]:
%pip install yfinance statsmodels openpyxl -q

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

We install the libraries from inside the notebook with `%pip` and not from the
terminal, because the notebook runs with its own Python and if we install from the
terminal it does not find the libraries.

## 1. Investment Universe

In [ ]:
tickers_mx = ["WALMEX.MX", "AC.MX", "FEMSAUBD.MX", "BIMBOA.MX", "ALSEA.MX"]
tickers_us = ["AAPL", "MSFT", "AMZN", "KO", "PG"]
tickers = tickers_mx + tickers_us

start_date = "2021-08-31"
end_date = "2026-08-31"

raw_data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)["Close"]
raw_data = raw_data[tickers]

print(raw_data.shape)
raw_data.head()

We use the same 10 stocks from Deliverable 1, downloaded with the yfinance API and
with adjusted closing prices, because some of these stocks pay dividends.

The universe is 5 Mexican stocks and 5 US stocks. We chose it this way for three
reasons:

- The firm invests in both markets, so the fund has to reflect that.
- The 10 stocks are not from the same sector. There is consumer staples (WALMEX,
  BIMBOA, KO, PG), beverages and bottling (AC, FEMSAUBD), restaurants (ALSEA) and
  technology (AAPL, MSFT, AMZN). If all of them were from the same sector they would
  go up and down together and there would be no diversification.
- The client has a moderate risk tolerance, so we mix defensive stocks like KO and PG
  with growth stocks like AAPL and AMZN, instead of going all in on one side.

In [ ]:
fondo = raw_data.ffill().dropna()
fondo.index = pd.to_datetime(fondo.index)

print("Days of prices per stock:")
print(fondo.count())
print()
print("Final shape:", fondo.shape)

With `.ffill()` we replace each missing value with the last known price, because when
a stock did not trade on a date the most recent price is the best information we have.
With `.dropna()` we remove the rows that are still empty after that, which are mostly
the days when one market was open and the other one was closed.

We print the count of days per stock to check that all of them downloaded well. If one
stock came out with very few days, its Beta and its Sharpe would be wrong and we would
have to check it before continuing.

In [ ]:
returns = fondo.pct_change().dropna()

fondo_return = returns.mean(axis=1)
fondo_return.name = "Fondo_Return"
fondo_index = (1 + fondo_return).cumprod()

print(returns.shape)
returns.head()

With `.pct_change()` we turn the prices into daily returns, and with `.mean(axis=1)`
we get the daily return of the fund assuming all 10 stocks have equal weight, the same
way we did it in Deliverable 1. `fondo_index` is the accumulated value of one peso
invested at the beginning.

In [ ]:
correlation = returns.corr()
print(correlation.round(2))

In [ ]:
# Average correlation inside Mexico, inside the US, and between the two markets.
# We take out the diagonal because a stock with itself is always 1.
corr_mx = correlation.loc[tickers_mx, tickers_mx]
corr_us = correlation.loc[tickers_us, tickers_us]
corr_mx_us = correlation.loc[tickers_mx, tickers_us]

avg_mx = (corr_mx.values.sum() - 5) / (5 * 5 - 5)
avg_us = (corr_us.values.sum() - 5) / (5 * 5 - 5)
avg_between = corr_mx_us.values.mean()

print(f"Average correlation between Mexican stocks: {avg_mx:.3f}")
print(f"Average correlation between US stocks:      {avg_us:.3f}")
print(f"Average correlation Mexico vs US:           {avg_between:.3f}")

The correlation is what tells us if the diversification is real or only apparent. If
the correlation between the two markets is lower than the correlation inside each
market, it means that when one market falls the other one does not necessarily fall
with it, and that is exactly what makes the portfolio less risky than its parts.

In [ ]:
# How much volatility we save by combining the 10 stocks
stock_volatility = returns.std() * np.sqrt(252)
average_volatility = stock_volatility.mean()
portfolio_volatility = fondo_return.std() * np.sqrt(252)

reduction = 1 - portfolio_volatility / average_volatility

print(stock_volatility.sort_values(ascending=False).round(4))
print()
print(f"Average volatility of the 10 stocks separately: {average_volatility:.2%}")
print(f"Volatility of the portfolio together:          {portfolio_volatility:.2%}")
print(f"Reduction from diversifying:                   {reduction:.2%}")

This is the number that justifies the universe. The average volatility of the stocks
one by one is higher than the volatility of the portfolio, and the difference is what
we win only by combining them. We did not give up return to get that, we only stopped
depending on a single stock.

## 2. Risk Metrics

Now we calculate the four metrics the deliverable asks for, plus two more that help
to explain the risk to the client: CVaR and the maximum drawdown.

In [ ]:
benchmarks = ["^GSPC", "^MXX", "^TNX"]   # S&P 500, IPC of Mexico, 10-year Treasury
macro_data = yf.download(benchmarks, start=start_date, end=end_date, auto_adjust=True)["Close"]
macro_data = macro_data.ffill().dropna()

sp500_return = macro_data["^GSPC"].pct_change().dropna()
ipc_return = macro_data["^MXX"].pct_change().dropna()

# The Treasury is already a rate in percent, so we only take the average and divide by 100
risk_free = macro_data["^TNX"].mean() / 100

print(f"Average 10-year Treasury rate: {risk_free:.2%}")

We need the benchmarks for two things: the Beta is measured against the market, and the
Sharpe ratio needs a risk free rate. We use the S&P 500 for the US stocks, the IPC for
the Mexican ones, and the 10-year Treasury as the risk free rate.

In [ ]:
# We go stock by stock and calculate every metric
rows = []

for ticker in tickers:
    r = returns[ticker]

    if ticker in tickers_mx:
        market = ipc_return
        market_name = "IPC"
    else:
        market = sp500_return
        market_name = "S&P 500"

    annual_return = (1 + r.mean()) ** 252 - 1
    annual_vol = r.std() * np.sqrt(252)
    beta = r.cov(market) / market.var()
    sharpe = (annual_return - risk_free) / annual_vol

    var_95 = np.percentile(r, 5)                 # historical VaR, worst 5% of days
    var_normal = r.mean() - 1.645 * r.std()      # parametric VaR, assuming normal
    cvar_95 = r[r <= var_95].mean()              # average loss when VaR is passed

    price_index = (1 + r).cumprod()
    drawdown = price_index / price_index.cummax() - 1
    max_drawdown = drawdown.min()

    rows.append({
        "Stock": ticker,
        "Market": market_name,
        "Ann. Return": annual_return,
        "Ann. Volatility": annual_vol,
        "Beta": beta,
        "Sharpe": sharpe,
        "VaR 95% hist.": var_95,
        "VaR 95% normal": var_normal,
        "CVaR 95%": cvar_95,
        "Max Drawdown": max_drawdown,
    })

risk_metrics = pd.DataFrame(rows).set_index("Stock")
risk_metrics.round(4)

What each metric is, in the order we calculated them:

- **Annualized volatility** is the standard deviation of the daily returns multiplied
  by the square root of 252, which are the trading days in a year. It measures how much
  the price moves, up or down.
- **Beta** is the covariance of the stock with its market divided by the variance of the
  market. A Beta of 1 means the stock moves the same as the market, above 1 it amplifies
  it and below 1 it cushions it. Each stock is measured against its own market, because
  comparing a Mexican stock against the S&P 500 would mix the currency effect with the
  market effect.
- **Sharpe ratio** is the return above the risk free rate divided by the volatility. It
  answers how much return we get for each unit of risk we take.
- **VaR 95%** is the loss that is only passed on 5% of the days. We calculate it in the
  two ways: the historical one, which sorts the real days and takes the percentile, and
  the parametric one, which assumes the returns are normal.
- **CVaR** is the average loss on those bad days. VaR says where the bad zone starts,
  CVaR says how bad it gets once you are in it.
- **Max drawdown** is the biggest fall from a peak. For a client like George it is the
  most useful number of all, because it is the loss he would have had to live through
  without selling.

In [ ]:
# The same metrics but for the whole portfolio.
# For the Beta we use a 50/50 mix of both markets, because the fund is half and half.
market_blend = (sp500_return + ipc_return).dropna() / 2

fondo_annual_return = (1 + fondo_return.mean()) ** 252 - 1
fondo_annual_vol = fondo_return.std() * np.sqrt(252)
fondo_beta = fondo_return.cov(market_blend) / market_blend.var()
fondo_sharpe = (fondo_annual_return - risk_free) / fondo_annual_vol

fondo_var = np.percentile(fondo_return, 5)
fondo_var_normal = fondo_return.mean() - 1.645 * fondo_return.std()
fondo_cvar = fondo_return[fondo_return <= fondo_var].mean()

fondo_drawdown = fondo_index / fondo_index.cummax() - 1
fondo_max_drawdown = fondo_drawdown.min()

print(f"Annualized return:   {fondo_annual_return:.2%}")
print(f"Annualized vol:      {fondo_annual_vol:.2%}")
print(f"Beta vs 50/50 mix:   {fondo_beta:.3f}")
print(f"Sharpe ratio:        {fondo_sharpe:.3f}")
print(f"VaR 95% historical:  {fondo_var:.2%}  daily")
print(f"VaR 95% normal:      {fondo_var_normal:.2%}  daily")
print(f"CVaR 95%:            {fondo_cvar:.2%}  daily")
print(f"Max drawdown:        {fondo_max_drawdown:.2%}")

The two VaR numbers do not come out the same and that is the point of calculating both.
The historical one uses the days that really happened and the parametric one assumes a
normal distribution. Returns have fatter tails than a normal, which means that the very
bad days happen more often than the formula predicts, so when the historical VaR is
worse than the parametric one, the normal is underestimating the risk.

For the client this translates to: on a normal day the fund does not lose more than the
VaR, but on the 5% of days that it does, the average loss is the CVaR.

## 3. Goodness of Fit of the Model

We build the same variables from Deliverable 1 and we compare three models to see which
one really explains the returns of the fund.

In [ ]:
# Technical variables, all shifted one day so we only use information from the past
tech = pd.DataFrame(index=fondo_return.index)
tech["Lag_Return"] = fondo_return.shift(1)
tech["SMA_50"] = fondo_index.rolling(window=50).mean().shift(1)
tech["SMA_200"] = fondo_index.rolling(window=200).mean().shift(1)

delta = fondo_index.diff()
gain = delta.where(delta > 0, 0.0)
loss = -delta.where(delta < 0, 0.0)
rs = gain.rolling(14).mean() / loss.rolling(14).mean()
tech["RSI_14"] = (100 - (100 / (1 + rs))).shift(1)

tech.tail()

In [ ]:
# Macro variables, also shifted one day
macro = pd.DataFrame(index=macro_data.index)
macro["SP500_lag1"] = macro_data["^GSPC"].pct_change().shift(1)
macro["IPC_lag1"] = macro_data["^MXX"].pct_change().shift(1)
macro["Treasury_lag1"] = macro_data["^TNX"].diff().shift(1)

model_data = pd.concat([fondo_return, tech, macro], axis=1).dropna()

print("Rows we can use for the models:", len(model_data))
model_data.head()

Everything is shifted one day with `.shift(1)` because if we used the same day's data
we would be predicting today with today's information, which in real life we would not
have. That is the mistake that makes a model look perfect in the notebook and lose money
in the market.

We join everything and use `.dropna()` once, so the three models are estimated on exactly
the same rows. This matters because AIC and BIC cannot be compared between models that
were fitted on a different number of observations.

In [ ]:
# Model 1: only technical variables
X1 = sm.add_constant(model_data[["Lag_Return", "SMA_50", "SMA_200", "RSI_14"]])
model_1 = sm.OLS(model_data["Fondo_Return"], X1).fit()

# Model 2: only macro variables
X2 = sm.add_constant(model_data[["SP500_lag1", "IPC_lag1", "Treasury_lag1"]])
model_2 = sm.OLS(model_data["Fondo_Return"], X2).fit()

# Model 3: everything together
X3 = sm.add_constant(model_data[["Lag_Return", "SMA_50", "SMA_200", "RSI_14",
                                 "SP500_lag1", "IPC_lag1", "Treasury_lag1"]])
model_3 = sm.OLS(model_data["Fondo_Return"], X3).fit()

comparison = pd.DataFrame({
    "Technical": [model_1.rsquared, model_1.rsquared_adj, model_1.aic, model_1.bic, len(X1.columns) - 1],
    "Macro": [model_2.rsquared, model_2.rsquared_adj, model_2.aic, model_2.bic, len(X2.columns) - 1],
    "Full": [model_3.rsquared, model_3.rsquared_adj, model_3.aic, model_3.bic, len(X3.columns) - 1],
}, index=["R2", "Adjusted R2", "AIC", "BIC", "Variables"])

comparison.round(4)

In [ ]:
print(model_3.summary())

How to read the three numbers:

- **R2** is the part of the movement of the fund that the model explains. It always goes
  up when you add variables, even if the variables are useless, so it cannot be used
  alone to choose a model.
- **AIC** and **BIC** fix that. They both take the fit and subtract a penalty for each
  variable, so a variable only pays off if it explains more than what it costs. The
  lower the AIC or BIC, the better the model. BIC punishes harder than AIC, so when the
  two disagree, BIC is choosing the simpler model.
- The **adjusted R2** does the same idea as AIC and BIC but on the R2 scale.

The important part is not which model wins but how small the R2 is. Daily returns are
almost impossible to predict, and that is what the weak form of the efficient market
hypothesis says: the past prices are already in today's price. A very low R2 here is not
a broken model, it is the honest result, and it is the reason our strategy in section 5
is based on trend and risk control and not on trying to guess tomorrow's return.

## 4. Simulation and Backtesting

Two different things. The simulation asks what could happen in the future, the backtest
asks what would have happened in the past.

### 4.1 Monte Carlo simulation

In [ ]:
np.random.seed(42)

initial_capital = 1000000
years = 10
days = years * 252
n_simulations = 2000

# We take the real daily returns of the fund and draw from them at random, with
# replacement. Each column of the matrix is one possible future of 10 years.
observed_returns = fondo_return.values
simulated = np.random.choice(observed_returns, size=(days, n_simulations), replace=True)

paths = initial_capital * np.cumprod(1 + simulated, axis=0)
final_values = paths[-1]

p5 = np.percentile(final_values, 5)
p25 = np.percentile(final_values, 25)
p50 = np.percentile(final_values, 50)
p75 = np.percentile(final_values, 75)
p95 = np.percentile(final_values, 95)

print(f"Starting capital: ${initial_capital:,.0f}   Horizon: {years} years   Scenarios: {n_simulations:,}")
print()
print(f"Worst 5%   (pessimistic): ${p5:,.0f}")
print(f"Percentile 25:            ${p25:,.0f}")
print(f"Median:                   ${p50:,.0f}")
print(f"Percentile 75:            ${p75:,.0f}")
print(f"Best 5%    (optimistic):  ${p95:,.0f}")
print()
print(f"Probability of ending with less than what was invested: {(final_values < initial_capital).mean():.1%}")
print(f"Probability of at least doubling the money:             {(final_values > 2 * initial_capital).mean():.1%}")

We use the returns that really happened instead of generating normal random numbers.
That way the simulation keeps the fat tails and the very bad days that the fund actually
had, and it does not invent a market that behaves better than the real one.

What we draw at random is the order of the days, not the returns themselves. The
assumption behind this is that the days are independent, which is not perfect because in
real life volatility comes in clusters, but for a 10-year horizon it is a reasonable
approximation.

The result is not one number, it is a range. That is the honest way to present it to the
client: with this fund and this horizon, this is the range of where he could end up, and
this is how likely each part of the range is.

### 4.2 Backtesting the strategy

In [ ]:
# Strategy: we are invested while the 50-day average is above the 200-day average,
# and we go to cash when it is below.
sma_50 = fondo_index.rolling(50).mean()
sma_200 = fondo_index.rolling(200).mean()

signal = (sma_50 > sma_200).astype(int)
signal = signal.shift(1)          # we trade with yesterday's signal, not today's

backtest = pd.DataFrame({"return": fondo_return, "signal": signal}).dropna()

# Transaction cost of 0.10% every time we enter or leave the market
cost = 0.001
trades = backtest["signal"].diff().abs().fillna(0)
backtest["strategy"] = backtest["signal"] * backtest["return"] - trades * cost

backtest["buy_and_hold_value"] = (1 + backtest["return"]).cumprod()
backtest["strategy_value"] = (1 + backtest["strategy"]).cumprod()

print("Number of trades:", int(trades.sum()))
backtest.tail()

Two details that change the result completely if you skip them:

- The signal is shifted one day with `.shift(1)`. Without that we would be buying on the
  same day we found out the averages crossed, which in real life is impossible and makes
  any strategy look brilliant.
- We charge a cost every time the position changes. A strategy that trades a lot can look
  good on paper and lose the whole advantage on commissions.

In [ ]:
# We compare the strategy against simply buying and holding
results = []

for name, r in [("Buy and hold", backtest["return"]), ("SMA 50/200 strategy", backtest["strategy"])]:
    total_return = (1 + r).prod() - 1
    annual_return = (1 + r.mean()) ** 252 - 1
    annual_vol = r.std() * np.sqrt(252)
    sharpe = (annual_return - risk_free) / annual_vol
    value = (1 + r).cumprod()
    max_dd = (value / value.cummax() - 1).min()
    var = np.percentile(r, 5)

    results.append({
        "Strategy": name,
        "Total Return": total_return,
        "Ann. Return": annual_return,
        "Ann. Volatility": annual_vol,
        "Sharpe": sharpe,
        "Max Drawdown": max_dd,
        "VaR 95%": var,
    })

comparison_bt = pd.DataFrame(results).set_index("Strategy")

# Days with a positive return. For the strategy we only count the days it was invested,
# because the days in cash are zero and would make the percentage look worse than it is.
invested = backtest["signal"] == 1
pct_positive_bh = (backtest["return"] > 0).mean()
pct_positive_st = (backtest.loc[invested, "strategy"] > 0).mean()

comparison_bt["% Positive Days"] = [pct_positive_bh, pct_positive_st]
comparison_bt["% Time in Market"] = [1.0, invested.mean()]

comparison_bt.round(4)

We separate the percentage of positive days from the percentage of time in the market on
purpose. The strategy spends part of the time in cash, and those days have a return of
exactly zero. If we counted them as flat days the percentage of positive days would come
out artificially low, when in reality the strategy was not even playing on those days.

## 5. Investment Strategy and Transaction Script

The strategy has to match the client. George is 41, has more than 10 years of horizon,
moderate tolerance and wants capital appreciation. It does not make sense to give him a
strategy that trades every day, and it does not make sense to give him one that has him
sitting through a 40% drawdown either.

### The rules

1. **Base allocation.** Equal weight in the 10 stocks, 10% each, half Mexico and half
   the US. Equal weight because we do not have a reason to think one of the 10 is going
   to beat the others, and concentrating would be taking a bet we cannot justify.
2. **Trend filter.** We stay invested while the 50-day average is above the 200-day
   average. When it crosses below, we move to cash. This is what protects the drawdown,
   which is the number that makes a client sell at the worst moment.
3. **Risk filter per stock.** Any stock with a Beta above 1.5 gets its weight cut in
   half, and the difference goes to the stocks with the lowest volatility. The client
   said moderate risk, so we do not carry positions that amplify the market that much.
4. **Rebalancing.** Quarterly, and only if a weight moved more than 5 percentage points
   away from its target. Rebalancing more often only generates commissions.
5. **Stop.** If the fund falls more than 20% from its peak we reduce to 50% invested and
   we wait for the trend to turn before going back in.

In [ ]:
# Script that generates the orders. It does not send anything to any broker.
capital = 1000000
target_weight = 0.10

current_prices = fondo.iloc[-1]
last_date = fondo.index[-1].date()

# We start from equal weights and apply the risk filter
weights = {}
for ticker in tickers:
    beta = risk_metrics.loc[ticker, "Beta"]
    if beta > 1.5:
        weights[ticker] = target_weight / 2
    else:
        weights[ticker] = target_weight

# What we took away from the risky stocks goes to the 3 least volatile ones
leftover = 1 - sum(weights.values())
safest = risk_metrics["Ann. Volatility"].sort_values().head(3).index

for ticker in safest:
    weights[ticker] = weights[ticker] + leftover / 3

print("Weights after the risk filter:")
for ticker in tickers:
    print(f"  {ticker:12s} {weights[ticker]:6.2%}   Beta {risk_metrics.loc[ticker, 'Beta']:.2f}")
print(f"\nTotal: {sum(weights.values()):.2%}")

In [ ]:
# Are we invested right now, according to the trend filter?
trend_is_up = sma_50.iloc[-1] > sma_200.iloc[-1]

print(f"SMA 50:  {sma_50.iloc[-1]:.4f}")
print(f"SMA 200: {sma_200.iloc[-1]:.4f}")
print("Signal:", "INVESTED" if trend_is_up else "IN CASH")
print()

orders = []

if trend_is_up:
    for ticker in tickers:
        amount = capital * weights[ticker]
        shares = int(amount / current_prices[ticker])
        orders.append({
            "Date": last_date,
            "Stock": ticker,
            "Action": "BUY",
            "Shares": shares,
            "Price": round(current_prices[ticker], 2),
            "Amount": round(shares * current_prices[ticker], 2),
            "Reason": "Equal weight with risk filter, trend up",
        })
else:
    for ticker in tickers:
        orders.append({
            "Date": last_date,
            "Stock": ticker,
            "Action": "SELL",
            "Shares": 0,
            "Price": round(current_prices[ticker], 2),
            "Amount": 0,
            "Reason": "SMA 50 below SMA 200, go to cash",
        })

orders = pd.DataFrame(orders)
orders.to_csv("orders.csv", index=False)

print("Orders generated and saved in orders.csv")
orders

In [ ]:
# HOW THESE ORDERS WOULD BE SENT TO A BROKER
#
# The code stays commented on purpose. This script generates and records the orders,
# but it does not execute them. Sending real money to the market is a decision for a
# person, not for a script that runs on its own.
#
# If it were connected, it would go to the paper trading account first, port 7497,
# never to the real account:
#
# from ib_insync import IB, Stock, MarketOrder
#
# ib = IB()
# ib.connect("127.0.0.1", 7497, clientId=1)     # 7497 = paper trading
#
# for i, order in orders.iterrows():
#     contract = Stock(order["Stock"], "SMART", "USD")
#     trade = MarketOrder(order["Action"], order["Shares"])
#     ib.placeOrder(contract, trade)
#
# ib.disconnect()

print("The orders are in orders.csv, ready for review before anyone executes them.")

The script leaves the orders written down with the date, the stock, the action, the
number of shares, the price and the reason. That last column is what makes it auditable:
in six months anyone can open the file and see why each position was taken.

## 6. Final Visualizations

In [ ]:
blue = "#2a78d6"
orange = "#eb6834"
green = "#1baf7a"

In [ ]:
# 1. The fund against the two markets
plt.figure(figsize=(11, 5))

plt.plot(fondo_index.index, fondo_index / fondo_index.iloc[0] * 100, color=blue, linewidth=2, label="Fund (10 stocks)")

sp_index = (1 + sp500_return).cumprod()
ipc_index = (1 + ipc_return).cumprod()
plt.plot(sp_index.index, sp_index / sp_index.iloc[0] * 100, color=orange, linewidth=2, label="S&P 500")
plt.plot(ipc_index.index, ipc_index / ipc_index.iloc[0] * 100, color=green, linewidth=2, label="IPC Mexico")

plt.title("Fund vs the markets (base = 100)")
plt.ylabel("Value of 100 invested")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Risk against return, stock by stock
plt.figure(figsize=(9, 6))

for ticker in tickers:
    x = risk_metrics.loc[ticker, "Ann. Volatility"]
    y = risk_metrics.loc[ticker, "Ann. Return"]
    color = orange if ticker in tickers_mx else blue
    plt.scatter(x, y, s=90, color=color, zorder=3)
    plt.annotate(ticker, (x, y), xytext=(6, 5), textcoords="offset points", fontsize=8)

plt.scatter(fondo_annual_vol, fondo_annual_return, s=200, color=green, marker="*", zorder=4)
plt.annotate("FUND", (fondo_annual_vol, fondo_annual_return), xytext=(8, -12),
             textcoords="offset points", fontsize=9, fontweight="bold")

plt.axhline(0, color="gray", linewidth=0.8)
plt.title("Risk vs return (orange = Mexico, blue = US, green = fund)")
plt.xlabel("Annualized volatility")
plt.ylabel("Annualized return")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The fund is the green star and it sits to the left of almost every stock, which means
less risk, while its return stays in the middle of the group. That is diversification
drawn on a chart: we did not get the return of the best stock, but we also did not take
the risk of any single one of them.

In [ ]:
# 3. Correlation between the 10 stocks
plt.figure(figsize=(8, 7))
plt.imshow(correlation, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Correlation")

plt.xticks(range(len(tickers)), tickers, rotation=90, fontsize=8)
plt.yticks(range(len(tickers)), tickers, fontsize=8)

for i in range(len(tickers)):
    for j in range(len(tickers)):
        plt.text(j, i, f"{correlation.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7,
                 color="white" if correlation.iloc[i, j] > 0.6 else "black")

# Line separating the Mexican stocks from the US ones
plt.axhline(4.5, color="white", linewidth=2)
plt.axvline(4.5, color="white", linewidth=2)

plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

In [ ]:
# 4. Distribution of the daily returns of the fund, with the VaR and the CVaR
plt.figure(figsize=(10, 5))
plt.hist(fondo_return, bins=60, color=blue, edgecolor="white")

plt.axvline(fondo_var, color=orange, linestyle="--", linewidth=2, label=f"VaR 95% = {fondo_var:.2%}")
plt.axvline(fondo_cvar, color="#8b2500", linestyle="--", linewidth=2, label=f"CVaR 95% = {fondo_cvar:.2%}")

plt.title("Daily returns of the fund")
plt.xlabel("Daily return")
plt.ylabel("Number of days")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 5. The Monte Carlo simulation
plt.figure(figsize=(11, 6))

years_axis = np.arange(days) / 252

plt.fill_between(years_axis, np.percentile(paths, 5, axis=1), np.percentile(paths, 95, axis=1),
                 color=blue, alpha=0.15, label="90% of the scenarios")
plt.fill_between(years_axis, np.percentile(paths, 25, axis=1), np.percentile(paths, 75, axis=1),
                 color=blue, alpha=0.3, label="50% of the scenarios")
plt.plot(years_axis, np.percentile(paths, 50, axis=1), color=blue, linewidth=2.5, label="Median")
plt.axhline(initial_capital, color=orange, linestyle="--", linewidth=2, label="Starting capital")

plt.title(f"{n_simulations:,} simulations of the fund over {years} years")
plt.xlabel("Years")
plt.ylabel("Portfolio value")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 6. Backtest: the strategy against buying and holding
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

axes[0].plot(backtest.index, backtest["buy_and_hold_value"], color=blue, linewidth=2, label="Buy and hold")
axes[0].plot(backtest.index, backtest["strategy_value"], color=orange, linewidth=2, label="SMA 50/200 strategy")
axes[0].set_title("Value of 1 invested")
axes[0].legend()
axes[0].grid(alpha=0.3)

dd_bh = backtest["buy_and_hold_value"] / backtest["buy_and_hold_value"].cummax() - 1
dd_st = backtest["strategy_value"] / backtest["strategy_value"].cummax() - 1

axes[1].fill_between(backtest.index, dd_bh, 0, color=blue, alpha=0.4, label="Buy and hold")
axes[1].fill_between(backtest.index, dd_st, 0, color=orange, alpha=0.4, label="Strategy")
axes[1].set_title("Drawdown, how far below the previous peak")
axes[1].set_ylabel("Fall from the peak")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

The second panel is the one that matters for this client. It is not enough for the
strategy to end higher, what matters is how deep the holes were on the way, because that
is what a real person has to live through without selling.

## 7. Export to Excel

In [ ]:
with pd.ExcelWriter("capital_analyst_metrics.xlsx") as writer:
    risk_metrics.to_excel(writer, sheet_name="Risk Metrics")
    correlation.to_excel(writer, sheet_name="Correlation")
    comparison.to_excel(writer, sheet_name="Model Comparison")
    comparison_bt.to_excel(writer, sheet_name="Backtest")
    orders.to_excel(writer, sheet_name="Orders", index=False)

    monte_carlo_table = pd.DataFrame({
        "Scenario": ["Worst 5%", "Percentile 25", "Median", "Percentile 75", "Best 5%"],
        "Value in 10 years": [p5, p25, p50, p75, p95],
    })
    monte_carlo_table.to_excel(writer, sheet_name="Monte Carlo", index=False)

print("Saved capital_analyst_metrics.xlsx")

## 8. Summary of Results

This last cell prints everything together. Copy this block to write the proposal document.

In [ ]:
print("=" * 62)
print("[8] SUMMARY OF RESULTS")
print("=" * 62)
print(f"Period: {fondo.index[0].date()} to {fondo.index[-1].date()}   ({len(fondo)} days)")
print()

print("--- 1. UNIVERSE ---")
print(f"Correlation inside Mexico:   {avg_mx:.3f}")
print(f"Correlation inside the US:   {avg_us:.3f}")
print(f"Correlation between markets: {avg_between:.3f}")
print(f"Average volatility of the stocks separately: {average_volatility:.2%}")
print(f"Volatility of the portfolio:                 {portfolio_volatility:.2%}")
print(f"Reduction from diversifying:                 {reduction:.2%}")
print()

print("--- 2. RISK METRICS OF THE FUND ---")
print(f"Annualized return: {fondo_annual_return:.2%}")
print(f"Annualized vol:    {fondo_annual_vol:.2%}")
print(f"Beta:              {fondo_beta:.3f}")
print(f"Sharpe:            {fondo_sharpe:.3f}")
print(f"VaR 95% hist:      {fondo_var:.2%}")
print(f"VaR 95% normal:    {fondo_var_normal:.2%}")
print(f"CVaR 95%:          {fondo_cvar:.2%}")
print(f"Max drawdown:      {fondo_max_drawdown:.2%}")
print(f"Risk free rate:    {risk_free:.2%}")
print()
print("Per stock:")
print(risk_metrics[["Ann. Return", "Ann. Volatility", "Beta", "Sharpe", "Max Drawdown"]].round(4))
print()

print("--- 3. MODELS ---")
print(comparison.round(4))
print()

print("--- 4. MONTE CARLO (10 years, $1,000,000) ---")
print(f"Worst 5%:      ${p5:,.0f}")
print(f"Percentile 25: ${p25:,.0f}")
print(f"Median:        ${p50:,.0f}")
print(f"Percentile 75: ${p75:,.0f}")
print(f"Best 5%:       ${p95:,.0f}")
print(f"Probability of losing money: {(final_values < initial_capital).mean():.1%}")
print(f"Probability of doubling:     {(final_values > 2 * initial_capital).mean():.1%}")
print()

print("--- 4.2 BACKTEST ---")
print(comparison_bt.round(4))
print(f"Number of trades: {int(trades.sum())}")
print()

print("--- 5. CURRENT SIGNAL ---")
print("Signal:", "INVESTED" if trend_is_up else "IN CASH")
print(f"Orders generated: {len(orders)}")
print()
print("=" * 62)